# Débruiteur U-Net — entraînement (Colab)

Autoencodeur **U-Net** entraîné sur **DIV2K** (images haute qualité) à retirer un
**mélange aléatoire de bruits** (gaussien, sel & poivre, Poisson, speckle, flou),
avec **Keras 3 + JAX**.


## 1. Récupérer le code du projet

In [ ]:
import os

REPO_URL = "https://github.com/tristan-reig/Projet_L3_REMAKE.git"
REPO_DIR = "/content/projet-ia-creative"

if not os.path.exists(REPO_DIR):
    !GIT_TERMINAL_PROMPT=0 git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

os.chdir(REPO_DIR)
print("Répertoire de travail :", os.getcwd())

## 2. Backend JAX

Pas de `tensorflow_datasets` ici : DIV2K est téléchargé directement, donc aucun conflit Protobuf.

In [ ]:
import os
os.environ["KERAS_BACKEND"] = "jax"

import keras
print("Keras :", keras.__version__, "| backend :", keras.backend.backend())

## 3. Paramètres

In [ ]:
IMG_SIZE   = 128
BATCH_SIZE = 32
EPOCHS     = 40
PATCHES_PER_IMAGE = 8   # patchs extraits par image DIV2K (plus = plus de données)

MODEL_NAME = "model_denoiser.keras"

## 4. (Optionnel) Monter Google Drive

Recommandé pour sauver les checkpoints.

In [ ]:
USE_DRIVE = True

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    SAVE_DIR = "/content/drive/MyDrive/projet-ia-creative/models"
else:
    SAVE_DIR = "/content/projet-ia-creative/models"

os.makedirs(SAVE_DIR, exist_ok=True)
MODEL_PATH = os.path.join(SAVE_DIR, MODEL_NAME)
print("Le modèle sera sauvegardé ici :", MODEL_PATH)

## 5. Préparer le dataset

Premier appel : téléchargement de DIV2K (~3.5 Go) puis mise en cache. On extrait plusieurs patchs aléatoires par image, chacun dégradé par un mélange de bruits tiré au hasard.

In [ ]:
from ml.denoiser.data import build_dataset

train_ds, val_ds = build_dataset(batch_size=BATCH_SIZE, patches_per_image=PATCHES_PER_IMAGE)
print("Datasets prêts.")

## 6. Visualiser des paires bruité / propre

Vérifie que les dégradations sont variées et réalistes.

In [ ]:
import matplotlib.pyplot as plt

noisy, clean = next(iter(train_ds))
plt.figure(figsize=(12, 6))
for i in range(4):
    plt.subplot(2, 4, i + 1)
    plt.imshow(noisy[i].numpy()); plt.title("Bruité", fontsize=9); plt.axis("off")
    plt.subplot(2, 4, i + 5)
    plt.imshow(clean[i].numpy()); plt.title("Propre", fontsize=9); plt.axis("off")
plt.tight_layout(); plt.show()

## 7. Construire le U-Net débruiteur

Encodeur / décodeur avec skip connections. La perte est la MAE (préserve les contours), la métrique le PSNR (qualité de reconstruction en dB).

In [ ]:
from ml.denoiser.model import build_denoiser, compile_model

model = build_denoiser(img_size=IMG_SIZE)
model = compile_model(model)
model.summary()

## 8. Entraîner

In [ ]:
from keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau

callbacks = [
    ModelCheckpoint(MODEL_PATH, monitor="val_psnr_metric", mode="max",
                    save_best_only=True, verbose=1),
    EarlyStopping(monitor="val_psnr_metric", mode="max", patience=8,
                  restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=4, min_lr=1e-6, verbose=1),
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks,
)

## 9. Courbes d'apprentissage

In [ ]:
import matplotlib.pyplot as plt

h = history.history
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, (tr, val, title) in zip(
    axes,
    [("loss", "val_loss", "Perte (MAE)"), ("psnr_metric", "val_psnr_metric", "PSNR (dB)")],
):
    ax.plot(h[tr], label="Train", linewidth=2)
    ax.plot(h[val], label="Val", linewidth=2, linestyle="--")
    ax.set_title(title); ax.set_xlabel("Époque"); ax.legend()
    ax.grid(True, linestyle="--", alpha=0.6)
fig.tight_layout(); plt.show()

print(f"Meilleur PSNR validation : {max(h['val_psnr_metric']):.2f} dB")

## 10. Tester le débruitage

On prend un patch propre, on le dégrade, et on compare bruité / débruité / original.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ml.denoiser.data import _degrade

noisy, clean = next(iter(val_ds))
restored = model.predict(noisy, verbose=0)

plt.figure(figsize=(12, 9))
for i in range(3):
    for j, (img, title) in enumerate([
        (noisy[i], "Bruité"), (restored[i], "Débruité"), (clean[i], "Original")
    ]):
        plt.subplot(3, 3, i * 3 + j + 1)
        plt.imshow(np.clip(img, 0, 1)); plt.title(title, fontsize=9); plt.axis("off")
plt.tight_layout(); plt.show()

## 11. Récupérer le modèle entraîné

In [ ]:
if not USE_DRIVE:
    from google.colab import files
    files.download(MODEL_PATH)
else:
    print("Modèle disponible sur Drive :", MODEL_PATH)